## BƯỚC 1: Setup

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
!pip install -q groq transformers torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from groq import Groq
import json
from datetime import datetime
from collections import deque
import re

print(f"🖥️ Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

🖥️ Device: cuda


## BƯỚC 2: Configuration

In [ ]:
# 🔧 CẤU HÌNH - Chỉnh theo setup của bạn

MODEL_PATH = "/content/drive/MyDrive/TrainAI/emotion_model_v5.2_counseling"
GROQ_API_KEY = "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"  # Nhập key từ console.groq.com

LABEL_MAP = {
    "sadness": 0, "joy": 1, "love": 2, "anger": 3,
    "fear": 4, "surprise": 5, "neutral": 6
}
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}

if not GROQ_API_KEY:
    print("⚠️ Vui lòng nhập Groq API key!")
    print("🔗 Lấy MIỄN PHÍ tại: https://console.groq.com/")
else:
    print("✅ Configuration OK")

✅ Configuration OK


## BƯỚC 3: Load Emotion Model

In [ ]:
"""
FIX CONFIG.JSON - Model V5.2
=============================
Chạy script này TRƯỚC KHI load model
"""

import json
import os

MODEL_PATH = "/content/drive/MyDrive/TrainAI/emotion_model_v5.2_counseling"
CONFIG_FILE = f"{MODEL_PATH}/config.json"

print("🔧 Fixing config.json...")

# Load existing config
try:
    with open(CONFIG_FILE, 'r') as f:
        config = json.load(f)
    print(f"✓ Loaded existing config")
except:
    print("✗ Config not found, creating new one")
    config = {}

# Fix config - Add required fields
fixed_config = {
    "architectures": [
        "XLMRobertaForSequenceClassification"
    ],
    "model_type": "xlm-roberta",  # CRITICAL: This was missing!
    "num_labels": 7,
    "id2label": {
        "0": "sadness",
        "1": "joy",
        "2": "love",
        "3": "anger",
        "4": "fear",
        "5": "surprise",
        "6": "neutral"
    },
    "label2id": {
        "sadness": 0,
        "joy": 1,
        "love": 2,
        "anger": 3,
        "fear": 4,
        "surprise": 5,
        "neutral": 6
    },
    # XLM-RoBERTa specific
    "attention_probs_dropout_prob": 0.25,
    "hidden_dropout_prob": 0.25,
    "hidden_size": 768,
    "intermediate_size": 3072,
    "max_position_embeddings": 514,
    "num_attention_heads": 12,
    "num_hidden_layers": 12,
    "type_vocab_size": 1,
    "vocab_size": 250002,
    "pad_token_id": 1,
    "bos_token_id": 0,
    "eos_token_id": 2,
}

# Merge with existing (keep any other fields)
for key, value in config.items():
    if key not in fixed_config:
        fixed_config[key] = value

# Save fixed config
with open(CONFIG_FILE, 'w') as f:
    json.dump(fixed_config, f, indent=2)

print(f"✅ Fixed config.json at: {CONFIG_FILE}")
print(f"   Model type: {fixed_config['model_type']}")
print(f"   Num labels: {fixed_config['num_labels']}")
print("\n🎯 Now you can load the model!")

🔧 Fixing config.json...
✓ Loaded existing config
✅ Fixed config.json at: /content/drive/MyDrive/TrainAI/emotion_model_v5.2_counseling/config.json
   Model type: xlm-roberta
   Num labels: 7

🎯 Now you can load the model!


In [ ]:
print("🔧 Loading emotion model...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
emotion_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
emotion_model.to(device)
emotion_model.eval()

print(f"✅ Emotion model loaded on {device}")
print(f"   Parameters: {sum(p.numel() for p in emotion_model.parameters()):,}")

🔧 Loading emotion model...


The tokenizer you are loading from '/content/drive/MyDrive/TrainAI/emotion_model_v5.2_counseling' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

✅ Emotion model loaded on cuda
   Parameters: 278,049,031


In [ ]:
def detect_emotion(text):
    """Dự đoán cảm xúc bằng model đã train"""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = emotion_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    # Primary emotion
    confidence, pred_id = torch.max(probs, dim=0)
    primary_emotion = ID2LABEL[pred_id.item()]

    # Top 3
    top3_probs, top3_ids = torch.topk(probs, 3)
    all_emotions = [
        (ID2LABEL[idx.item()], prob.item())
        for idx, prob in zip(top3_ids, top3_probs)
    ]

    return {
        'primary_emotion': primary_emotion,
        'confidence': confidence.item(),
        'all_emotions': all_emotions
    }

print("✅ Emotion detector ready!")

✅ Emotion detector ready!


## BƯỚC 4: Setup Groq Advice Generator

In [ ]:
# Initialize Groq
groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client initialized")

✅ Groq client initialized


In [ ]:
# System prompts cho từng emotion
EMOTION_PROMPTS = {
    "sadness": """Bạn là chuyên gia tư vấn tâm lý ấm áp.

Khi người dùng buồn:
- Đồng cảm sâu sắc, không minimize cảm xúc
- Đề xuất hoạt động nhẹ nhàng (đi dạo, nghe nhạc, gặp bạn...)
- Nhắc nhở tự chăm sóc bản thân
- Khuyến khích chia sẻ nếu cần

Dùng "mình" và "bạn", 3-5 câu, TIẾNG VIỆT.""",

    "joy": """Bạn là chuyên gia tư vấn tâm lý ấm áp.

Khi người dùng vui:
- Chúc mừng và chia vui
- Khuyến khích duy trì năng lượng tích cực
- Gợi ý chia sẻ niềm vui với người khác
- Nhắc nhở trân trọng khoảnh khắc

Dùng "mình" và "bạn", 3-5 câu, TIẾNG VIỆT.""",

    "love": """Bạn là chuyên gia tư vấn tâm lý ấm áp.

Khi người dùng thể hiện tình yêu/biết ơn:
- Trân trọng cảm xúc đẹp
- Khuyến khích thể hiện tình cảm
- Gợi ý hành động cụ thể (viết thư, nói lời cảm ơn...)
- Nhắc nhở nuôi dưỡng mối quan hệ

Dùng "mình" và "bạn", 3-5 câu, TIẾNG VIỆT.""",

    "anger": """Bạn là chuyên gia tư vấn tâm lý ấm áp.

Khi người dùng giận:
- Thừa nhận cảm xúc là bình thường
- Gợi ý cách xả stress lành mạnh (thể thao, viết nhật ký...)
- Đề xuất giải quyết vấn đề khi đã bình tĩnh
- Tránh hành động bốc đồng

Dùng "mình" và "bạn", 3-5 câu, TIẾNG VIỆT.""",

    "fear": """Bạn là chuyên gia tư vấn tâm lý ấm áp.

Khi người dùng lo lắng/sợ:
- Đồng cảm với nỗi lo
- Gợi ý kỹ thuật giảm stress (thở sâu, viết ra...)
- Đề xuất chuẩn bị cụ thể
- Nhắc nhở năng lực bản thân

Dùng "mình" và "bạn", 3-5 câu, TIẾNG VIỆT.""",

    "surprise": """Bạn là chuyên gia tư vấn tâm lý ấm áp.

Khi người dùng bất ngờ:
- Thừa nhận cảm giác choáng váng
- Gợi ý xử lý (nếu tiêu cực) hoặc tận hưởng (nếu tích cực)
- Khuyến khích phản ứng thích hợp

Dùng "mình" và "bạn", 3-5 câu, TIẾNG VIỆT.""",

    "neutral": """Bạn là chuyên gia tư vấn tâm lý ấm áp.

Khi người dùng trung lập:
- Tôn trọng trạng thái bình thường
- Gợi ý hoạt động tích cực nhẹ nhàng
- Không áp đặt cảm xúc

Dùng "mình" và "bạn", 3-5 câu, TIẾNG VIỆT."""
}

print("✅ Emotion-specific prompts loaded")

✅ Emotion-specific prompts loaded


In [ ]:
def generate_advice(text, emotion_result, conversation_history=None):
    """Sinh lời khuyên context-aware"""

    emotion = emotion_result['primary_emotion']
    confidence = emotion_result['confidence']

    # Context từ history
    history_context = ""
    if conversation_history and len(conversation_history) > 0:
        history_context = "\n\nLịch sử gần đây:"
        for conv in conversation_history[-2:]:  # Last 2
            history_context += f"\n- User: {conv['input']}"
            history_context += f"\n  Bot: {conv['advice'][:80]}..."

    # User prompt
    user_prompt = f"""Phân tích và đưa lời khuyên:

**Cảm xúc:** {emotion.upper()} ({confidence:.0%})

**Câu nói:**
\"{text}\"
{history_context}

**Yêu cầu:**
1. NHẮC LẠI ngữ cảnh cụ thể (công việc, tình yêu, gia đình, học tập...)
2. Thấu hiểu và đồng cảm
3. Lời khuyên CỤ THỂ, THỰC TẾ
4. 3-5 câu
5. TIẾNG VIỆT

Lời khuyên:"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": EMOTION_PROMPTS.get(emotion, EMOTION_PROMPTS["neutral"])},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_tokens=600
        )

        advice = response.choices[0].message.content.strip()
        # Clean up
        advice = re.sub(r'\*\*.*?\*\*', '', advice)
        advice = re.sub(r'^(Lời khuyên:|Advice:)', '', advice).strip()

        return advice

    except Exception as e:
        print(f"❌ API Error: {e}")
        # Fallback
        fallback = {
            "sadness": "Mình hiểu bạn đang buồn. Hãy cho phép bản thân được nghỉ ngơi và chia sẻ với người thân nhé.",
            "joy": "Thật tuyệt khi bạn vui! Hãy trân trọng khoảnh khắc này.",
            "love": "Cảm xúc yêu thương rất đẹp. Đừng ngại thể hiện tình cảm nhé.",
            "anger": "Mình hiểu bạn giận. Hãy thở sâu và xử lý khi bình tĩnh.",
            "fear": "Lo lắng là bình thường. Hãy chuẩn bị tốt và tin vào mình.",
            "surprise": "Bất ngờ nhỉ! Hãy dành thời gian xử lý cảm xúc.",
            "neutral": "Ngày bình thường cũng tốt. Có cần chia sẻ gì không?"
        }
        return fallback.get(emotion, "Cảm ơn đã chia sẻ!")

print("✅ Advice generator ready!")

✅ Advice generator ready!


## BƯỚC 5: Complete System

In [ ]:
class EmotionAdviceSystem:
    """Hệ thống hoàn chỉnh"""

    def __init__(self):
        self.conversation_history = deque(maxlen=5)
        print("✅ System initialized!")

    def process(self, text):
        """Xử lý và sinh lời khuyên"""
        # 1. Detect emotion
        emotion_result = detect_emotion(text)

        # 2. Generate advice
        advice = generate_advice(
            text,
            emotion_result,
            list(self.conversation_history)
        )

        # 3. Result
        result = {
            'input': text,
            'emotion': emotion_result['primary_emotion'],
            'confidence': emotion_result['confidence'],
            'all_emotions': emotion_result['all_emotions'],
            'advice': advice,
            'timestamp': datetime.now().isoformat()
        }

        # 4. Save history
        self.conversation_history.append({
            'input': text,
            'emotion': emotion_result['primary_emotion'],
            'advice': advice
        })

        return result

    def display(self, result):
        """Hiển thị kết quả"""
        print("\n" + "=" * 80)
        print(f"📝 USER: {result['input']}")
        print("-" * 80)
        print(f"😊 EMOTION: {result['emotion'].upper()} ({result['confidence']:.1%})")

        top3_str = ', '.join([f"{e}({s:.0%})" for e, s in result['all_emotions']])
        print(f"   Top 3: {top3_str}")

        print("-" * 80)
        print(f"💬 ADVICE:")
        print(f"   {result['advice']}")
        print("=" * 80)

# Initialize
system = EmotionAdviceSystem()
print("\n🎉 System ready to use!")

✅ System initialized!

🎉 System ready to use!


---
## 🧪 TEST HỆ THỐNG
---

In [ ]:
# Test comprehensive
test_cases = [
    "Hôm nay trời đẹp quá, mọi điều may mắn đến với em",
    "Em buồn lắm, bạn trai bỏ em rồi",
    "Lo lắng về buổi phỏng vấn ngày mai quá",
    "Anh yêu em nhiều lắm",
    "Tức quá, sếp đối xử bất công",
    "Không ngờ đậu đại học luôn",
    "Đang đi làm bình thường thôi",
    "I'm so happy today!",
    "I failed the exam, feeling devastated",
]

print("\n" + "=" * 80)
print("🧪 TESTING ALL EMOTIONS")
print("=" * 80)

for text in test_cases:
    result = system.process(text)
    system.display(result)
    print()


🧪 TESTING ALL EMOTIONS

📝 USER: Hôm nay trời đẹp quá, mọi điều may mắn đến với em
--------------------------------------------------------------------------------
😊 EMOTION: JOY (99.8%)
   Top 3: joy(100%), love(0%), neutral(0%)
--------------------------------------------------------------------------------
💬 ADVICE:
   Mình rất vui khi nghe bạn nói vậy! Dường như hôm nay là một ngày tuyệt vời với bạn, và mọi thứ đang diễn ra theo hướng tích cực. Mình không rõ ngữ cảnh cụ thể là gì, nhưng có thể đó là một ngày may mắn trong công việc, tình yêu, hoặc học tập. Dù thế nào, mình cũng muốn nhắc bạn hãy trân trọng và tận hưởng từng khoảnh khắc này, vì những ngày như thế này thật sự đáng quý. Hãy chia sẻ niềm vui của mình với người khác, và đừng quên để lại những kỷ niệm đẹp trong tâm trí.


📝 USER: Em buồn lắm, bạn trai bỏ em rồi
--------------------------------------------------------------------------------
😊 EMOTION: SADNESS (100.0%)
   Top 3: sadness(100%), love(0%), joy(0%)
----------

## 💬 INTERACTIVE CHAT MODE

In [ ]:
def chat_mode():
    """Chat tương tác"""
    print("\n💬 CHAT MODE - Nhập 'quit' để thoát\n")

    while True:
        try:
            user_input = input("👤 Bạn: ").strip()

            if not user_input:
                continue

            if user_input.lower() in ['quit', 'q', 'exit', 'thoát']:
                print("\n👋 Tạm biệt! Chăm sóc bản thân nhé!")
                break

            # Process
            result = system.process(user_input)

            # Display short
            print(f"\n🤖 Bot ({result['emotion']}, {result['confidence']:.0%}):")
            print(f"   {result['advice']}\n")

        except KeyboardInterrupt:
            print("\n\n👋 Tạm biệt!")
            break

# Uncomment để chat
# chat_mode()